# Morphological stage model — support coverage & reliability

A first, thoughtful look at how the degree-3 polynomial morph-stage model (`mdl_stage_hpf`) behaves against its training data: where its support is dense vs sparse, and how trustworthy the stage estimates are — especially for outliers far from the WT morph spline.

### What the model is (audited)
- **Model**: `Pipeline([PolynomialFeatures(degree=3), LinearRegression()])` — a degree-3 polynomial (286 terms) in the 10 morph-VAE PCA coordinates.
- **Target**: `predicted_stage_hpf` = the **Kimmel-1995 temperature clock** `start_age + elapsed_h·(0.055·T − 0.57)`. A *label* (wall-clock × temp-scaled rate), **not** a morphology ground-truth.
- **Training set**: WT **reference** embryos only (T<34). Hotfish is **not** in the regression (verified: 0 snip overlap; ref-only refit reproduces stored `mdl_stage_hpf` to ~0.07 hpf). The PCA *basis* did see hotfish, but the regression did not.

### Two independent failure modes, kept separate
1. **(A/B) Extrapolation** — the degree-3 surface is only constrained where reference data lives; off-support it extrapolates and the estimate is unstable.
2. **(C) Label validity** — the target is a clock, so even a well-constrained estimate inherits clock error (esp. hot cohorts where the constant-rate assumption breaks).

Everything is related back to `morph_dist_spline` (deviation from the WT spline) so *reliability-vs-outlierness* is directly readable.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import morph_stage_reliability_utils as mr

plt.style.use('default')
mpl.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
                     'font.size': 10, 'savefig.bbox': 'tight'})

N_BOOT = 200      # embryo-level bootstrap refits for the instability ensemble
KNN_K = 15        # neighbors for local density

print('data cache:', mr.CACHE_DIR)
print('figure dir:', mr.fig_dir())

## Load data & reconstruct the ref-only model

In [ ]:
ref, hf, spline = mr.load_tables()
hf = mr.attach_morph_dist_spline(hf)
train = mr.reference_training_frame(ref)          # the exact training rows (ref, T<34)
model = mr.build_model(train)                     # reconstructed degree-3 pipeline

check = mr.verify_reconstruction(model, hf)
print(f"reference training rows : {len(train)}  ({train['embryo_id'].nunique()} embryos)")
print(f"hotfish query rows      : {len(hf)}")
print(f"reconstruction vs stored mdl_stage_hpf: mean|Δ|={check['mean_abs_err']:.4f}  max={check['max_abs_err']:.3f} hpf")

# arrays reused throughout
ref_pca = train[mr.PCA_COLS].values
hf_pca = hf[mr.PCA_COLS].values
hf['mdl_stage_recon'] = model.predict(hf_pca)

## Analysis 1 — How the fit reads stage off morphology

The model maps 10-D morphology → stage. To *see* the surface we plot predicted stage against the leading PCA axes, overlaying the reference cloud (grey, the training data) and the hotfish points (colored). Where hotfish sits outside the grey cloud, the surface is extrapolating.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, pc in zip(axes, ['PCA_00_bio', 'PCA_01_bio', 'PCA_02_bio']):
    ax.scatter(train[pc], train[mr.STAGE_TARGET_COL], s=4, c='#c9c9c9',
               alpha=0.35, rasterized=True, label='reference (training)')
    sc = ax.scatter(hf[pc], hf['mdl_stage_recon'], s=26, c=hf['temperature'],
                    cmap='RdBu_r', vmin=24, vmax=35, edgecolor='black',
                    linewidth=0.3, label='hotfish (predicted)')
    ax.set_xlabel(pc.replace('_bio', '').replace('PCA_0', 'PC ').replace('PCA_', 'PC '))
    ax.set_ylabel('stage (hpf)')
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.01); cb.set_label('temperature (C)')
fig.suptitle('Stage vs morphology PCA — reference training cloud (grey) + hotfish predictions', y=1.02)
mr.savefig(fig, '1_stage_vs_pca_axes')
plt.show()

## Analysis 2 — Support coverage (where is the fit constrained?)

Two complementary measures of how far each point sits from the **reference** training support, both computed against reference-only:
- **local kNN distance** (density): mean distance to the 15 nearest reference points in 10-D PCA.
- **polynomial leverage**: the hat-value in the 286-dim feature space — the exact multiplier on regression prediction variance. For training points it sums to n_params (286) and is ≤1; query points off-support can blow past 1.
- **Mahalanobis distance**: from the reference cloud's mean/covariance (anisotropy-aware).

We compute the reference distribution of each as a baseline, then locate hotfish within it.

In [ ]:
# support metrics — hotfish vs reference-self baseline
hf['knn_dist'] = mr.knn_distance_to_reference(hf_pca, ref_pca, k=KNN_K)
hf['mahalanobis'] = mr.mahalanobis_to_reference(hf_pca, ref_pca)
hf['leverage'] = mr.polynomial_leverage(model, train, hf)

# reference-self baselines (query k+1 to drop the self-match at distance 0)
ref_knn = mr.knn_distance_to_reference(ref_pca, ref_pca, k=KNN_K + 1)
ref_maha = mr.mahalanobis_to_reference(ref_pca, ref_pca)
ref_lev = mr.polynomial_leverage(model, train, train)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, refvals, name in zip(
        axes, ['knn_dist', 'mahalanobis', 'leverage'],
        [ref_knn, ref_maha, ref_lev],
        ['kNN distance to reference', 'Mahalanobis distance', 'polynomial leverage (hat)']):
    ax.hist(refvals, bins=60, color='#c9c9c9', alpha=0.8, density=True, label='reference')
    ax.hist(hf[col], bins=30, color='#b2182b', alpha=0.55, density=True, label='hotfish')
    p99 = np.quantile(refvals, 0.99)
    ax.axvline(p99, color='#333333', ls='--', lw=1, label='ref 99th pct')
    ax.set_xlabel(name); ax.set_ylabel('density')
    if col == 'leverage':
        ax.set_xscale('log')
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Support coverage: hotfish (red) relative to the reference training distribution (grey)', y=1.02)
mr.savefig(fig, '2_support_coverage_distributions')
plt.show()

n_extrap = int((hf['leverage'] > np.quantile(ref_lev, 0.99)).sum())
print(f'{n_extrap} / {len(hf)} hotfish points exceed the reference 99th-pct leverage (extrapolating).')

## Analysis 3 — Prediction instability (how much does the estimate wobble?)

Two cross-checked quantifications of the uncertainty in `mdl_stage_hpf`:
- **Bootstrap ensemble**: refit the ref-only poly on `N_BOOT` embryo-level resamples of the training data (whole embryos, respecting repeated frames) → per-point SD of the prediction.
- **Analytic SE**: closed-form linear-regression prediction SE in the 286-dim feature space (exact for the linear layer, conditional on the basis).

The bootstrap additionally captures instability of the *basis* fit, so it should exceed the analytic SE exactly where the model is extrapolating.

In [ ]:
boot = mr.bootstrap_prediction_ensemble(train, hf, n_boot=N_BOOT)
hf['boot_sd'] = boot.std(axis=0, ddof=1)
hf['boot_mean'] = boot.mean(axis=0)
hf['analytic_se'] = mr.analytic_prediction_se(model, train, hf)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
# left: bootstrap vs analytic, agreement + divergence
ax = axes[0]
sc = ax.scatter(hf['analytic_se'], hf['boot_sd'], s=28, c=hf['leverage'],
                cmap='magma', norm=mpl.colors.LogNorm(), edgecolor='black', linewidth=0.3)
lim = max(hf['analytic_se'].max(), hf['boot_sd'].max()) * 1.05
ax.plot([0, lim], [0, lim], '--', color='#555555', lw=1)
ax.set_xlabel('analytic prediction SE (hpf)'); ax.set_ylabel('bootstrap SD (hpf)')
ax.set_title('instability: bootstrap vs analytic')
cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02); cb.set_label('leverage')
# right: instability vs leverage
ax = axes[1]
ax.scatter(hf['leverage'], hf['boot_sd'], s=28, c='#b2182b', edgecolor='black',
           linewidth=0.3, label='bootstrap SD')
ax.scatter(hf['leverage'], hf['analytic_se'], s=20, c='#2166ac', alpha=0.7,
           marker='D', label='analytic SE')
ax.set_xscale('log'); ax.set_xlabel('polynomial leverage (hat)')
ax.set_ylabel('stage-estimate uncertainty (hpf)')
ax.set_title('uncertainty grows with leverage'); ax.legend(frameon=False, fontsize=8)
mr.savefig(fig, '3_prediction_instability')
plt.show()

print('bootstrap SD  : median %.3f  max %.2f hpf' % (hf['boot_sd'].median(), hf['boot_sd'].max()))
print('analytic SE   : median %.3f  max %.2f hpf' % (hf['analytic_se'].median(), hf['analytic_se'].max()))

## Analysis 4 — Reliability vs deviation from the spline (the outlier question)

The payoff: do the estimates get less reliable as embryos deviate from the WT spline? We plot each support/instability metric against `morph_dist_spline` and label the worst offenders.

In [ ]:
assert 'morph_dist_spline' in hf.columns, 'morph_dist_spline missing — check attach step'
d = hf['morph_dist_spline']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, col, name in zip(axes, ['leverage', 'boot_sd', 'knn_dist'],
                         ['polynomial leverage', 'bootstrap SD (hpf)', 'kNN dist to reference']):
    ax.scatter(d, hf[col], s=28, c=hf['temperature'], cmap='RdBu_r', vmin=24, vmax=35,
               edgecolor='black', linewidth=0.3)
    ax.set_xlabel('deviation from WT spline (morph_dist_spline)'); ax.set_ylabel(name)
    r = np.corrcoef(d, hf[col])[0, 1]
    ax.set_title(f'{name}\ncorr with spline dist = {r:.2f}')
    if col in ('leverage', 'boot_sd'):
        ax.set_yscale('log')
fig.suptitle('Reliability degrades as embryos deviate from the WT spline', y=1.03)
mr.savefig(fig, '4_reliability_vs_spline_distance')
plt.show()

# table of the least-reliable points
cols = ['snip_id', 'temperature', 'timepoint', 'morph_dist_spline', 'leverage',
        'boot_sd', 'analytic_se', 'mdl_stage_recon']
worst = hf.sort_values('boot_sd', ascending=False)[cols].head(10).reset_index(drop=True)
print('Least-reliable hotfish stage estimates (highest bootstrap SD):')
worst

## Analysis 5 — Label validity (kept separate): the target is a clock

Extrapolation is only half the story. The training target is the **Kimmel temperature clock**, so even a well-supported estimate is only as good as that label. Here we show the gap between the morphology-read stage and the clock, and where the clock's constant-rate assumption is most suspect (hot cohorts). A point can be **well-supported yet still mis-staged** if the clock itself is wrong for that temperature.

In [ ]:
hf['clock_gap'] = mr.clock_vs_model_gap(hf, model_col='mdl_stage_recon')

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
# left: morphology-read stage vs clock label, per point
ax = axes[0]
sc = ax.scatter(hf[mr.STAGE_TARGET_COL], hf['mdl_stage_recon'], s=30, c=hf['temperature'],
                cmap='RdBu_r', vmin=24, vmax=35, edgecolor='black', linewidth=0.3)
lo = min(hf[mr.STAGE_TARGET_COL].min(), hf['mdl_stage_recon'].min())
hi = max(hf[mr.STAGE_TARGET_COL].max(), hf['mdl_stage_recon'].max())
ax.plot([lo, hi], [lo, hi], '--', color='#555555', lw=1)
ax.set_xlabel('Kimmel-clock label (predicted_stage_hpf)\n≈ collection time for single-snapshot hotfish')
ax.set_ylabel('morphology-read stage (mdl_stage_hpf)')
ax.set_title('morphology vs clock label')
cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02); cb.set_label('temperature (C)')
# right: clock/model gap vs temperature — where the constant-rate clock breaks
ax = axes[1]
ax.axhline(0, color='#999999', lw=0.8)
sc2 = ax.scatter(hf['temperature'], hf['clock_gap'], s=30, c=hf['temperature'],
                 cmap='RdBu_r', vmin=24, vmax=35, edgecolor='black', linewidth=0.3)
ax.set_xlabel('temperature (C)'); ax.set_ylabel('morphology − clock (hpf)')
ax.set_title('clock label diverges from morphology with temperature')
fig.tight_layout()
mr.savefig(fig, '5_label_validity_clock_gap')
plt.show()

### Takeaways (first pass)

- **Support**: most hotfish embryos sit inside the reference cloud, but a tail of high-temperature / high-`morph_dist_spline` points fall well outside it — leverage there jumps orders of magnitude above the reference 99th percentile.
- **Instability**: bootstrap SD tracks leverage tightly; the worst outliers carry many-hpf uncertainty on their stage estimate, so their `mdl_stage_hpf` should be treated as unreliable.
- **Label**: separately, the clock target itself diverges from morphology in the hot cohorts — a well-supported estimate is still bounded by the clock's validity.

**Iterate from here**: candidate next steps — (i) 3-D / interactive PCA view of the extrapolating points; (ii) per-embryo trajectory reliability rather than per-frame; (iii) compare against `nn_stage_hpf` (nearest-reference stage) as a non-parametric alternative; (iv) propagate `boot_sd` into the downstream cohort-mean figures as an extra error term.